# Building a Transformer From Scratch

Decoder-only (GPT-style) transformer, following `learning_material/transformer-roadmap.md`.

In [ ]:
## loading the dataset
with open('../data/dataset.txt', 'r', encoding='utf-8') as f:
    dataset = f.read()

print(dataset[:30])

In [ ]:
## vocabulary: every unique character -> an integer
uchars = sorted(list(set(dataset)))
vocab_size = len(uchars)
print(vocab_size, 'unique characters')
print(''.join(uchars))

stoi = {ch: i for i, ch in enumerate(uchars)}  # char -> int
itos = {i: ch for i, ch in enumerate(uchars)}  # int -> char

encode = lambda s: [stoi[c] for c in s]
decode = lambda n: ''.join([itos[i] for i in n])

print(encode('Hi, if you are reading this, hire me'))
print(decode(encode('Hi, if you are reading this, hire me')))

In [ ]:
import torch

data = torch.tensor(encode(dataset), dtype=torch.long)

## train / validation split
n = int(0.95 * len(data))   # 95% train, 5% validation
train_data = data[:n]
val_data = data[n:]
print(len(train_data), 'train tokens |', len(val_data), 'val tokens')

In [ ]:
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32      # sequences processed in parallel
block_size = 64      # maximum context length
max_iters = 3000
eval_interval = 300
eval_iters = 100
learning_rate = 1e-3
n_embd = 128         # embedding dimension
n_head = 4           # attention heads
n_layer = 4          # transformer blocks
dropout = 0.1
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(1337)

In [ ]:
## grab a batch of (input, next-char target) pairs
def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))  # random start positions
    x = torch.stack([d[i:i + block_size] for i in ix])          # inputs
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])  # targets = input shifted by 1
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('inputs ', tuple(xb.shape))
print('targets', tuple(yb.shape))